In [ ]:
# install necessary packages
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

# Import the Anthropic client and set up the model
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Not the most efficient way to do this, but it demonstrates the streaming API
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model, max_tokens=1000, messages=messages, temperature=0.5, stream=True
)

for event in stream:
    print(event)

In [ ]:
# Instead of manually parsing the events, we can use the SDK's simplified streaming interface that extracts just the text content
with client.messages.stream(model=model, max_tokens=1000, messages=messages) as stream:
    for text in stream.text_stream:
        print(text, end="")

# Get the complete message for database storage
final_message = stream.get_final_message()